In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
from openai import OpenAI
from datasets import load_dataset, DatasetDict
from sklearn.metrics import precision_recall_fscore_support, accuracy_score
import numpy as np
from transformers import AutoModelForSequenceClassification, AutoTokenizer, TrainingArguments, Trainer, EarlyStoppingCallback, DataCollatorWithPadding
from pgkd import PGKD
import wandb

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("distilbert/distilbert-base-uncased")

In [ ]:
OPENAI_API_KEY = os.environ["OPENAI_API_KEY"]

client = OpenAI(api_key=OPENAI_API_KEY)

## Load in dataset, subsample and tokenize

#### TODO breakout into own file

In [ ]:
def split_dataset(dataset, text_column="headline", label_column="label", label_name="category", train_size=800, val_size=200, seed=42):
    """
    Split a Huggingface dataset into train, validation, and test sets.
    
    Args:
        dataset: datasets.Dataset
        train_size (int): Number of samples for training set
        val_size (int): Number of samples for validation set
        seed (int): Random seed for reproducibility
        
    Returns:
        dict: Dictionary containing train, validation, and test datasets
    """
    
    # If dataset has multiple splits, use the 'train' split as the full dataset
    if isinstance(dataset, dict):
        dataset = dataset['test']
    
    # Calculate total size and test size
    total_size = len(dataset)
    test_size = total_size - train_size - val_size
    
    if test_size <= 0:
        raise ValueError(f"Requested split sizes exceed dataset size ({total_size})")
     
    # Select and rename columns
    dataset = dataset.select_columns([text_column, label_column, label_name])
    dataset = dataset.rename_columns({
        text_column: "text",
        label_column: "label",
        label_name: "label_name"
    })
    
    # Create splits
    splits = dataset.train_test_split(
        train_size=train_size + val_size,
        test_size=test_size,
        seed=seed,
        shuffle=True
    )
    
    # Further split the train portion into train and validation
    train_val = splits['train'].train_test_split(
        train_size=train_size,
        test_size=val_size,
        seed=seed,
        shuffle=True
    )
    
    return {
        'train': train_val['train'],
        'validation': train_val['test'],
        'test': splits['test']
    }

ds = load_dataset("khalidalt/HuffPost")

dataset = DatasetDict(split_dataset(ds))

label2id = {sample["label_name"]: sample["label"] for sample in dataset["test"]}
id2label = {i: label for label, i in label2id.items()}

In [ ]:
def preprocess_function(examples):
    return tokenizer(
        examples["text"], 
        padding=True,  # Add this
        truncation=True, 
        return_tensors="pt"  # Explicitly return PyTorch tensors
    )

tokenized_data = dataset.map(preprocess_function, batched=True,
                             remove_columns=["text", "label_name"])
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [ ]:
taxonomy = [{"label": key, "label_name": id2label[key]} for key in sorted(list(id2label.keys()))]

In [ ]:
import json
# system message
system_message = """
You will receive a dictionary of category labels. For each category, provide a brief, clear description.
Your response must be valid JSON matching the input structure with an added "description" field.
"""

# Prepare the user message with the categories
user_message = f"Please add descriptions to these categories: {json.dumps(taxonomy, indent=2)}"

In [ ]:
from response_formats import Taxonomy

In [ ]:
response = client.beta.chat.completions.parse(
            model="gpt-4o-mini", 
            messages=[
                {"role": "system", "content": system_message},
                {"role": "user", "content": user_message}
            ],
            response_format=Taxonomy,
)

In [ ]:
taxonomy = [class_info.dict() for class_info in response.choices[0].message.parsed.classes]

## Initialise model
For speed and compute, we use distilbert for demos. The original paper uses bert-base

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained("distilbert/distilbert-base-uncased", num_labels=len(id2label), id2label=id2label, label2id=label2id)

In [ ]:
def compute_metrics(eval_pred):
    """
    Compute evaluation metrics for multi-class classification.
    
    Args:
        eval_pred: tuple containing:
            predictions: numpy array of shape (n_samples,) containing predicted labels
            labels: numpy array of shape (n_samples,) containing true labels
    
    Returns:
        dict: Dictionary containing evaluation metrics
    """
    predictions, labels = eval_pred
    
    # For multi-class, predictions are logits, so we need to convert to predicted classes
    predictions = np.argmax(predictions, axis=1)
    
    # Calculate precision, recall, and F1 score
    # Set average='weighted' for multi-class scenarios
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, 
        predictions, 
        average='weighted'
    )
    
    # Calculate accuracy
    acc = accuracy_score(labels, predictions)
    
    return {
        'eval_accuracy': acc,
        'eval_f1': f1,
        'eval_precision': precision,
        'eval_recall': recall,
    }

## Define training arguments

In [ ]:
training_args = TrainingArguments(
    output_dir="model_0",
    learning_rate=2e-5,
    per_device_train_batch_size=64,
    per_device_eval_batch_size=64,
    num_train_epochs=30,
    eval_strategy="steps",
    save_strategy="steps",
    eval_steps = 50,
    save_total_limit = 5,
    load_best_model_at_end=True,
    push_to_hub=False,
    metric_for_best_model='eval_loss'
)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_data["train"],
    eval_dataset=tokenized_data["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks = [EarlyStoppingCallback(early_stopping_patience=5)]
)

In [ ]:
trainer.train()

In [ ]:
trainer.state.log_history[-2]

### PGKD

### Load Model_0

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained("model_0/checkpoint-390/", num_labels=len(id2label), id2label=id2label, label2id=label2id)

In [ ]:
pgkd = PGKD(
            tokenizer, 
            model, 
            num_labels=len(id2label), 
            initial_dataset=tokenized_data["train"], 
            val_dataset=tokenized_data["validation"],
            dataset_class_taxonomy=taxonomy,
            openai_client=client,
            num_kd_steps=5,
            id2label=id2label,
            label2id=label2id,
            batch_size=8
            
)

In [ ]:
pgkd.train()

In [ ]:
for tokens in pgkd.train_dataset["input_ids"]:
    print(len([token for token in tokens if token == 0]))